# Phase 2 : XAI (SHAP) & Agent LLM 🧠

Objectif :
1. **Ouvrir la boîte noire** de notre modèle XGBoost grâce à SHAP (Values de Shapley).
2. **Générer une explication en langage naturel** avec l'Agent LLM (Ollama / Mistral).

*Pré-requis : avoir lancé `ollama serve` et téléchargé le modèle (`ollama pull mistral`) en arrière-plan.*

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from loguru import logger
from src.data.loader import DataLoader
from src.data.preprocessor import FraudPreprocessor
from src.models.trainer import ModelTrainer
from src.xai.explainer import FraudExplainer
from src.agent.llm_client import FraudAgent

## 1. Chargement du dataset et des modèles entraînés

In [ ]:
# Le Test Set n'a jamais été vu par le modèle pendant le SMOTE ni pendant l'entraînement
loader = DataLoader()
df = loader.load()

cols_to_drop = ['organization', 'transaction_id', 'user_id', 'transaction_timestamp']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

X_train, X_val, X_test, y_train, y_val, y_test = loader.get_splits(df_clean)

# Re-chargement des modèles depuis le HD
preprocessor = FraudPreprocessor.load(Path("../models/preprocessor.joblib"))
xgb_trainer = ModelTrainer.load(Path("../models/xgboost.joblib"), model_name="xgboost")

## 2. Instantiation de l'Explainer SHAP

In [ ]:
# On lui donne le modèle Scikit-learn brut et le nom des features transformées par le preprocessor
explainer = FraudExplainer(
    model=xgb_trainer.model,
    feature_names=preprocessor.feature_names,
    model_type="tree"
)
logger.info("Explainer SHAP Prêt !")

## 3. Sélection de deux transactions pour l'analyse
On prend une Transaction Légitime et une Transaction Frauduleuse du **Test Set**.

In [ ]:
# 1. Légitime
idx_legit = y_test[y_test == 0].index[0]
x_legit_raw = df.loc[idx_legit]   # Avec metadata (id, date etc.)
x_legit_features = X_test.loc[[idx_legit]]

# 2. Fraude
idx_fraud = y_test[y_test == 1].index[0]
x_fraud_raw = df.loc[idx_fraud]
x_fraud_features = X_test.loc[[idx_fraud]]

# Application du preprocessing pour le modèle
x_legit_proc = preprocessor.transform(x_legit_features)
x_fraud_proc = preprocessor.transform(x_fraud_features)

## 4. SHAP Waterfall & Top Features (Exemple = Fraude)

In [ ]:
# Prédiction
proba_fraud = xgb_trainer.predict_proba(x_fraud_proc)[0, 1]
print(f"Probabilité de fraude prédite : {proba_fraud:.1%}")

# SHAP Waterfall
fig = explainer.plot_waterfall(x_fraud_proc, transaction_id=x_fraud_raw["transaction_id"])

# Extraction des Top Features (formattées pour le LLM)
shap_values = explainer.explain_instance(x_fraud_proc)
top_features = explainer.get_top_features(shap_values, n=5)

print("\n📌 Top Features pour le LLM :")
for f in top_features:
    print(f)


## 5. Explication NLP par l'Agent IA
Ici, l'IA (Mistral via Ollama) va recevoir les données et les valeurs SHAP pour rédiger une analyse.

In [ ]:
# Vérification de l'état d'Ollama
llm_agent = FraudAgent(model="mistral")

if llm_agent.health_check():
    print("Agent LLM en ligne ! Génération en cours...\n")
    
    explication = llm_agent.explain(
        transaction=x_fraud_raw.to_dict(),
        fraud_probability=proba_fraud,
        top_features=top_features,
        threshold=0.35 # Le seuil optimal qu'on a vu en Phase 1
    )
    
    print("="*50)
    print("🤖 EXPLICATION GÉNÉRÉE :\n")
    print(explication)
    print("="*50)
else:
    print("⚠️ Ollama n'est pas lancé. Fallback sur les règles métiers basiques utilisé.")
    
    # Fallback si pas de LLM
    explication = llm_agent.explain(
        transaction=x_fraud_raw.to_dict(),
        fraud_probability=proba_fraud,
        top_features=top_features
    )
    print(explication)